# int8 GEMM on a GPU: run every kernel on Colab

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. The last cell downloads `results.zip` (CSV, device info, charts).

Takes about 5 minutes. The CPU reference check at 4096 uses sampled entries, so nothing here takes long.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvcc --version | tail -2

In [ ]:
%cd /content
!rm -rf int8-gemm-cuda
!git clone -q https://github.com/elvisamaya/int8-gemm-cuda.git
%cd int8-gemm-cuda
!git log --oneline | head -3

## Build and check correctness
Every kernel is compared exactly against a CPU result on awkward shapes (sizes that are not multiples of the tile).

In [ ]:
!make ARCH=sm_75 2>&1 | tail -3
!make test

## Benchmark
Each kernel at 512³ … 4096³, timed with CUDA events and checked again.

In [ ]:
!make bench

## CPU baseline and comparison with the systolic-array accelerator

In [ ]:
!python3 scripts/cpu_baseline.py results
!python3 scripts/compare_accelerator.py results

## Charts

In [ ]:
!make plots
from IPython.display import Image, display
for p in ['gops_by_kernel', 'gops_vs_size', 'roofline']:
    display(Image(f'results/{p}.png'))

## Optional: Nsight Compute profile
Skip if it errors - Colab sometimes blocks the GPU performance counters.

In [ ]:
!ncu --version 2>/dev/null && ncu --kernel-name regex:'gemm_' --launch-count 6 \
   --section SpeedOfLight --section MemoryWorkloadAnalysis \
   ./build/gemm_bench --sizes 1024 --out /tmp/ncu_run > results/ncu_summary.txt 2>&1; tail -40 results/ncu_summary.txt

## Download the results

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')